In [16]:
import json
import pandas as pd
OUT = "../../data/dataset/applets/applets_synt_new_final.jsonl"

#records = [replace_nan_with_none(r) for r in df_plus.to_dict(orient="records")]

df=pd.read_json(OUT,orient='records',lines=True)
#df.to_json(OUT,orient="records",lines=True,force_ascii=False)
import json
import pandas as pd
OUT_r = "../../data/dataset/applets/applets_real_new.jsonl"

#records = [replace_nan_with_none(r) for r in df_plus.to_dict(orient="records")]

df_r=pd.read_json(OUT_r,orient='records',lines=True)
import ast 
df['service_names']=df['service_names'].apply( lambda x: ast.literal_eval(x))
df_r['service_names']=df_r['service_names'].apply( lambda x: ast.literal_eval(x))

df["rule_signature"] = df.apply(
    lambda r: "T:" + "|".join(sorted(r.trigger_apis)) + "_A:" + "|".join(sorted(r.action_apis)),
    axis=1
)
df["op"]=df.apply(
    lambda r: set(r.trigger_apis) | set(r.action_apis),
    axis=1
)
df_r["op"]=df_r.apply(
    lambda r: set(r.trigger_apis) | set(r.action_apis),
    axis=1
)
df=df.drop_duplicates(subset="rule_signature",inplace=False)

MIN_R = 5
MAX_R = 10
K = len(df)-300

S_rem = set()
rest_idx = df.index.copy()

while len(rest_idx) > K:
    services_left = set().union(*df.loc[rest_idx, "service_names"]) - S_rem
    candidates = []

    for s in services_left:
        mask_use_s = df.loc[rest_idx, "service_names"].apply(lambda S: s in S)
        n_removed = mask_use_s.sum()

        # servizio valido solo se rimuove una quantità "moderata"
        if n_removed < MIN_R or n_removed > MAX_R:
            continue
        if len(rest_idx) - n_removed < K:
            continue

        candidates.append((s, n_removed))

    if not candidates:
        break

    # scegli quello che rimuove un numero di righe più vicino a MAX_R
    best_s = sorted(candidates, key=lambda x: -x[1])[0][0]

    S_rem.add(best_s)
    rest_idx = df.loc[rest_idx][
        ~df.loc[rest_idx, "service_names"].apply(lambda S: best_s in S)
    ].index

df_train   = df.loc[rest_idx].copy()
df_test = df.drop(rest_idx).copy()

services_real = set().union(*df_r["service_names"]) if len(df_r) > 0 else set() 
services_train = set().union(*df_train["service_names"]) if len(df_train) > 0 else set() 
services_test = set().union(*df_test["service_names"]) if len(df_test) > 0 else set()
op_train=set().union(*df_train["op"]) if len(df_train) > 0 else set() 


# verifica 
print("intersezione TRAIN ∩ S_rem:", services_train & S_rem) 
print("intersezione TEST ∩ S_rem:", len(services_test & S_rem)) 
print("intersezione REAL ∩ S_rem:", len(services_real & S_rem)) 
print("intersezione REAL ∩ TRAIN:", len(services_real & services_train)) 
print("len TRAIN:", len(df_train)) 
print("len TEST:", len(df_test)) 
print("len servizi rimossi REM:", len(S_rem))

# quante righe in df_train usano almeno un servizio in S_rem
n_righe_con_srem = df_train["service_names"].apply(
    lambda S: len(set(S) & S_rem) > 0
).sum()

print("Righe train che usano Srem", n_righe_con_srem)

# quante righe in df_test usano almeno un servizio in S_rem
n_righe_con_srem = df_test["service_names"].apply(
    lambda S: len(set(S) & S_rem) > 0
).sum()

print("Righe test che usano Srem", n_righe_con_srem)

# quante righe in df_real usano almeno un servizio in df_test
n_righe_con_srem = df_r["op"].apply(
    lambda S: len(set(S) & op_train) > 0
).sum()

print("Righe real che usano train", n_righe_con_srem)


intersezione TRAIN ∩ S_rem: set()
intersezione TEST ∩ S_rem: 37
intersezione REAL ∩ S_rem: 13
intersezione REAL ∩ TRAIN: 85
len TRAIN: 2246
len TEST: 298
len servizi rimossi REM: 37
Righe train che usano Srem 0
Righe test che usano Srem 298
Righe real che usano train 341


In [17]:
TRAIN = "../../data/dataset/applets/applets_synt_train.jsonl"
TEST = "../../data/dataset/applets/applets_synt_test.jsonl"


In [18]:
df_train.drop(columns=['require_filter_code','rule_signature','op','filter_code'], inplace=True)
df_train.rename(columns={'filter_code_gpt':'filter_code'}, inplace=True)
df_test.drop(columns=['require_filter_code','rule_signature','op','filter_code'], inplace=True)
df_test.rename(columns={'filter_code_gpt':'filter_code'}, inplace=True)

In [19]:
df_train.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)
df_train['row_index']=df_train.index.to_list()
df_test['row_index']=df_test.index.to_list()

In [28]:
import json

records = [r for r in df_train.to_dict(orient="records")]
for r in records:
    r['user_intent_example']=r['user_intent_example'].strip("\"")
    r['user_intent_example']=r['user_intent_example'].strip("'")

records_test = [r for r in df_test.to_dict(orient="records")]
for r in records:
    r['user_intent_example']=r['user_intent_example'].strip("\"")
    r['user_intent_example']=r['user_intent_example'].strip("'")

In [29]:

from utils.load import load_json_or_jsonl
from pathlib import Path
from llm_utility.prompts.prompt_in import make_prompt, get_prompt_config

PROMPT_CFG_ZS = get_prompt_config('catalog_full')
PROMPT_CFG_FT = get_prompt_config('minimal')
TRIGGER = Path("../../data/ifttt_catalog/triggers.json")
ACTION  =Path("../../data/ifttt_catalog/actions.json")
triggers = load_json_or_jsonl(TRIGGER)
actions  = load_json_or_jsonl(ACTION)
trigger_index = {t["api_endpoint_slug"]: t for t in triggers}
action_index = {a["api_endpoint_slug"]: a for a in actions}
for r in records:
    r['instruct_zs']=make_prompt(r,trigger_index,action_index,PROMPT_CFG_ZS)
    r['instruct_ft']=make_prompt(r,trigger_index,action_index,PROMPT_CFG_FT)
for r in records_test:
    r['instruct_zs']=make_prompt(r,trigger_index,action_index,PROMPT_CFG_ZS)
    r['instruct_ft']=make_prompt(r,trigger_index,action_index,PROMPT_CFG_FT)

df_train = pd.DataFrame(records)
df_test = pd.DataFrame(records_test)
import pandas as pd



In [30]:
df_train.to_json(TRAIN,orient='records',lines=True,force_ascii=False)
df_test.to_json(TEST,orient='records',lines=True,force_ascii=False)